# Imports
Chamando libs padrões e nesse caso os modelos do insight face.

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from numpy.linalg import norm
from insightface.app import FaceAnalysis

c:\Users\Usuário\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Insightface e o Buffalo 1
Esse "buffalo_l" um pacote com várias ONNX como o detector facial, reconhecimento e landmarks. Sendo ele um modelo "ONNX" ou seja "Open Neural Network Exchange".
Esse pacote é baseado no treinamento de um modelo e então na exportação para essa ONNX para que ela possa ser usada em qualquer ambiente. Como em outros frameworks eu só poderia usar os modelos nele mesmo então criaram esse tipo de framework para que ele pudesse ser passado para diversos sistemas.

Na prática o que ocorre é isso: Modelo é feito em pytorch por exemplo. Logo após isso ocorre um torch.onnx.export, gerando um modelo.onnx que pode ser executado em multiplas linguagens.

Dentro desse arquivo são geradas múltiplas coisas como a arquitetura, pesos, bias e operações.
Sendo tudo salvo como um grafo DAG, que por exemplo segue a ordem:

Modelo treinado em PyTorch(Um exemplo mas ele podia ser treinado em qualquer lugar)
↓
torch.onnx.export(...)
↓
Geração de modelo.onnx
↓
Execução em múltiplas linguagens e runtimes

O que ocorre quando um modelo onnx é chamado?
Ele baixa arquivos que são reconhecidos como as funcionalidades, como o detector facial ou o reconhecedor facial e então ele roda no ONNX Runtime que é o motor que executa modelos ONNX. Lendo o grafo e otimizando para CPU/GPU. 

Exemplos comumente usados de ONNX: Yolo ( detecção de objetos e criação de bounding boxes ), Whisper( trasncrição de audio ), PaddleOCR( ler texto de uma imagem ) e etc.

O buffalo_1 não é o único modelo, ele é o escolhido por possuir a melhor precisão mas existem outros dois ONNX mais leves e menos precisos.

## buffalo_l 
Ele é composto de quatro ONNX:

### • det_10g.onnx - Detecção facial
Baseado em SCRFD detectando os  pontos principais do rosto.
Foi treinado com ReLU.

### • w600k_r50.onnx — Reconhecimento Facial (ArcFace)
Baseado em ResNet-50
Transformando o rosto em um embedding de 512 dimensões.
Foi treinado usando PreLu que é uma modificação da ReLu que força o modelo a não ficar "viciado" corrigindo o erro do dying ReLu.

### • 2d106det.onnx — Landmarks Faciais 
Detecta pontos faciais detalhados como mandíbula, sobrancelhas e etc.

### • genderage.onnx — Idade e Gênero 
Calcula genero/idade aproximada

Foi treinado com ReLu.

SoftMax/Cross-Entropy -> Gênero
L1 Loss -> Idade

# Cosine Similarity
Calcula a similaridade de cosseno. Nesse caso eu uso ela para ver se uma pessoa é a que está no banco de dados ou não.

In [2]:
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0)
# Usando esse id como 0 pois estou usando minha gpu, se fosse -1 seria minha cpu

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Usuário/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Usuário/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Usuário/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Usuário/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Usuário/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112,

# Extração de embedding

Na minha pesquisa eu não vou precisar fazer esse passo porém como aqui foi feito um teste utilizando as operações do pacote do insight vision.

In [3]:
def getEmbedding(imagePath):

    # Lê e cria uma matriz 
    img = cv2.imread(imagePath)
    
    if img is None:
        return None

    # Extraindo os embeddings e detecta os rostos   
    faces = app.get(img)

    if len(faces) == 0:
        return None

    face = faces[0]
    # face = max(
    #     faces,
    #     key=lambda f: (
    #         f.bbox[2] - f.bbox[0]
    #     ) * (
    #         f.bbox[3] - f.bbox[1]
    #     )
    # )

    # Nesse caso eu tenho uma foto possivelmente mais de um rosto detectado então
    # eu pegaria o maior rosto mas no meu teste eu estou fazendo com fotos com 
    # apenas um rosto
    return face.embedding

# Criação do banco

Aqui é onde eu coloco o meu banco de dados e acontece a preparação desses dados. 

In [4]:
def buildDatabase(databasePath):

    #Criação de um dicionario vazio
    database = {}

    for file in os.listdir(databasePath):

        path = os.path.join(
            databasePath,
            file
        )

        if not os.path.isfile(path):
            continue

        embedding = getEmbedding(path)

        if embedding is None:
            continue
        
        # Essa parte tira a extensao do nome, tive um leve problema com isso e
        # resolvi esse erro dessa forma, os nomes agora estão com a visualizacao
        # melhor
        name = os.path.splitext(file)[0]

        database[name] = embedding

    return database

# Reconhecimento facial

In [5]:
def recognize_face(
    imagePath,
    database,
    threshold=0.5
):
    # Extraindo o embedding relacionado a imagem do teste de reconhecimento
    embeddingTeste = getEmbedding(imagePath)

    if embeddingTeste is None:
        return None

    bestMatch = None
    bestScore = -1

    # Percorrendo pessoas do banco
    for name, embeddingDB in database.items():

        # Faço a operação cosine similarity e descubro quem tem a maior similaridade
        # uso ela pois dentro da onnx e um arcface e ele possui uma similaridade
        # melhor com o cosine similarity apesar dos embeddings terem sido normalizados
        score = cosine_similarity(
            embeddingTeste,
            embeddingDB
        )

        # Mostro a similaridade com as pessoas do banco de dados
        print(
            f"{name} -> Similaridade: "
            f"{score:.4f}"
        )

        # Salvo quem foi o maior, podia fazer um max mas assim é mais facil do que
        # guardar todo mundo
        if score > bestScore:
            secondBest = bestScore
            secondBestName = bestMatch

            bestScore = score
            bestMatch = name

    print("\n\n\n")

    if bestScore >= threshold:

        # Coloquei o threshold para poder dizer que reconheci ou não a pessoa
        # Coloquei o threshold para poder dizer que reconheci ou não a pessoa
        print(f"Reconhecido: {bestMatch}\nScore: {bestScore:.4f}\n")
        print(f"Segunda pessoa com melhor score: {secondBestName}\nScore: {secondBest:.4f}\n")

        return bestMatch

    else:
        print(f"Pessoa desconhecida\nMelhor score: {bestScore:.4f}")

        return None

# Reconhecimento apenas para visualização
Aqui eu fiz o que normalmente aparece em vídeos e tutoriais sobre reconhecimento faciais de fazer uma bounding box visual assim como tem nesse vídeo: [link](https://www.youtube.com/watch?v=y-D1tReryGA)

In [6]:
def recognize_and_draw(
    image_path,
    arquiveType,
    database,
    threshold=0.5
):

    if(arquiveType==0): img = cv2.imread(image_path)
    else: img = image_path
    faces = app.get(img)

    # Percorrendo todos os rostos 
    for face in faces:

        embedding = face.embedding
        bestMatch = None
        bestScore = -1

        for name, embDB in database.items():
            # Mesma coisa do Recognition
            score = cosine_similarity(
                embedding,
                embDB
            )

            if score > bestScore:

                bestScore = score
                bestMatch = name

        if bestScore < threshold:
            bestMatch = "Desconhecido"

        # Converte o bounding box para um inteiro
        bbox = face.bbox.astype(int)

        x1, y1, x2, y2 = bbox
        # Faço um quadrado usando as coordenadas extraidas do bounding box
        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        texto = (
            f"{bestMatch} "
            f"({bestScore:.2f})"
        )

        # o cv2 nao permite usar fontes diferentes
        cv2.putText(
            img,
            texto,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_DUPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    return img

In [7]:
def saveDatabase(
    database,
    filename="faces_db.npy"
):

    np.save(
        filename,
        database
    )

def loadDatabase(
    filename="faces_db.npy"
):

    database = np.load(
        filename,
        allow_pickle=True
    ).item()

    return database

DATABASE_PATH = "bancoRostos"

database = buildDatabase(
    DATABASE_PATH
)

saveDatabase(database)

database = loadDatabase()

c:\Users\Usuário\AppData\Local\Programs\Python\Python313\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


In [ ]:
recognize_face(
    "Hannah.png",
    database,
    threshold=0.5
)

result = recognize_and_draw(
    "Hannah.png",
    0,
    database,
    threshold=0.5
)

cv2.imshow(
    "Resultado",
    result
)

cv2.waitKey(0)
cv2.destroyAllWindows()

AdamSandler -> Similaridade: 0.0500
ConanGray -> Similaridade: -0.0104
Hannah Carla L S N -> Similaridade: 0.6963
JoaoPedro -> Similaridade: 0.0974
Mainha -> Similaridade: 0.0324
Tata -> Similaridade: -0.0036
Vitoria -> Similaridade: 0.1552




Reconhecido: Hannah Carla L S N
Score: 0.6963

Segunda pessoa com melhor score: AdamSandler
Score: 0.0500



In [12]:
folder = "testeRostos"

valid_ext = [".png", ".jpg", ".jpeg"]
index = 0

images = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if os.path.splitext(f)[1].lower() in valid_ext
]

while True:

    imagePath = images[index]
    result = recognize_and_draw(
        imagePath,
        0,
        database,
        threshold=0.5
    )

    filename = os.path.basename(imagePath)

    cv2.imshow("Resultado", result)

    cv2.setWindowTitle(
        "Resultado",
        filename
    )

    key = cv2.waitKeyEx(0)
    if cv2.getWindowProperty("Resultado", cv2.WND_PROP_VISIBLE) < 1:
        break

    # -->
    elif key == 2555904:
        print("Direita")
        if index < len(images) - 1:
            index += 1

    # <--
    elif key == 2424832:
        print("Esquerda")
        if index > 0:
            index -= 1

cv2.destroyAllWindows()

Direita
Direita
Direita
Direita


In [10]:
cap = cv2.VideoCapture(
    0,
    cv2.CAP_DSHOW
)

while True:

    ret, frame = cap.read()
    if not ret:
        break

    # Reconhecimento facial
    result = recognize_and_draw(
        frame,
        1,
        database,
        threshold=0.5
    )

    cv2.imshow(
        "Reconhecimento Facial",
        result
    )

    key = cv2.waitKey(1)
    if key == 27:
        break

    if cv2.getWindowProperty("Reconhecimento Facial",cv2.WND_PROP_VISIBLE) < 1: break

# Finaliza
cap.release()

cv2.destroyAllWindows()

 # PARA O MEU COMMIT NO GITHUB EXCLUIREI AS PASTAS "BANCOSROSTOS" , "FACES" E "TESTEROSTOS" POIS NÃO TENHO AUTORIZAÇÃO PARA A POSTAGEM DAS FOTOS.